# Harry Kane — Goal-Mouth Placement

Where did Kane's goals actually cross the line? Every goal of his 2025/26
campaign for **Bayern Munich & England** binned into a 3 × 6 goal frame, in the
style of an Opta Analyst goal-mouth map, with a body-part breakdown and the
penalty / non-penalty split.

**Viz by [@oladeeayo](https://github.com/oladeeayo)** — rendered at 800 dpi to
[`figures/kane_goal_map.png`](figures/kane_goal_map.png) (plus a vector PDF).

![Harry Kane goal-mouth placement](figures/kane_goal_map.png)

Everything below is reproducible: the notebook reads
[`goals.csv`](goals.csv) (one row per goal, from WhoScored & Sofascore Opta
event data), registers the bundled DM Sans / Karla fonts and draws the whole
figure in matplotlib — no other dependencies.

**Contents** · [`goals.csv`](goals.csv) — the data ·
[`fonts/`](fonts) — DM Sans & Karla (SIL Open Font License) ·
[`figures/`](figures) — the rendered output · this notebook.


In [ ]:
from __future__ import annotations

import csv
import time
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Circle, Polygon, Rectangle
from IPython.display import Image, display

HERE = Path.cwd()
assert (HERE / "goals.csv").exists(), "run this notebook from its kane-goal-mouth/ folder"
(HERE / "figures").mkdir(exist_ok=True)

OUT_PNG = HERE / "figures" / "kane_goal_map.png"
OUT_PDF = HERE / "figures" / "kane_goal_map.pdf"

# --- fonts ship with the repo, so the render is identical anywhere -------
DISPLAY = "DM Sans"   # titles, numbers
BODY = "Karla"        # labels, captions
for _ttf in sorted((HERE / "fonts").glob("*.ttf")):
    font_manager.fontManager.addfont(str(_ttf))

# --- goal frame, in Opta goal-mouth units --------------------------------
Y_LEFT_POST = 45.2    # Opta GoalMouthY at the left post (shooter's left)
Y_RIGHT_POST = 54.8   # ... and at the right post (7.32 m wide)
Z_GROUND = 0.0
Z_BAR = 38.0          # Opta GoalMouthZ at the crossbar (2.44 m tall)

GOAL_W, GOAL_H = 7.32, 2.44          # real-world size of the frame, metres
COLS, ROWS = 6, 3

# --- canvas --------------------------------------------------------------
FIG_W, FIG_H, DPI = 14.8, 9.2, 800

AX_X, AX_Y, AX_W = 0.078, 0.27, 0.64
X_MARGIN, Y_MARGIN = 0.75, 0.62
X_SPAN = GOAL_W + 2 * X_MARGIN       # 8.82
Y_SPAN = GOAL_H + Y_MARGIN + 0.5     # 3.56
AX_H = AX_W * FIG_W * Y_SPAN / X_SPAN / FIG_H   # keep the axes on data aspect

# right-hand sidebar (body-part breakdown)
SIDE_X = 0.752                       # left edge of the sidebar block
ICON_CX = 0.8165                     # icon centres
ICON_SIZE = 0.90                     # inches, square — identical for all three
ICON_ROWS = (0.578, 0.462, 0.346)    # even pitch between the three rows
TEXT_CX = 0.8824                     # centre line of the number/label stack

# summary badges under the goal
BADGE_CX = (0.213, 0.398, 0.583)
BADGE_Y = 0.205
BADGE_W = 1.35                       # hexagon width, inches

# --- palette -------------------------------------------------------------
BG = "#F1F0EE"
INK = "#2E2E2E"
INK_SOFT = "#6A6A6A"
FRAME = "#3D3D3D"
RED = "#D6202B"
RULE = "#C9C7C4"

CELL_CMAP = LinearSegmentedColormap.from_list(
    "opta_red",
    ["#FFFFFF", "#FBE3E3", "#F7B9BB", "#EF7C81", "#E1313B", "#B81620"],
)


## The data — and the one thing that had to be worked out first

`goals.csv` carries, per goal, two Opta goal-mouth coordinates (WhoScored calls
them `GoalMouthY` / `GoalMouthZ`; Sofascore's shotmap exposes the same numbers
as `goalMouthCoordinates.{y,z}` — verified identical on Bayern 5-1 Köln):

| column | is | meaning |
|---|---|---|
| `goal_mouth_x` | Opta **GoalMouthY** | lateral position where the ball crossed the goal line (0–100 pitch frame) |
| `goal_mouth_y` | Opta **GoalMouthZ** | height at which it crossed the line |

**Neither column is a length in metres**, so the frame has to be calibrated
before anything can be drawn. This uses the community-standard conversion
(Rteta's `standardize_opta_y` / `standardize_opta_z`, which matches kloppy's
Opta pitch spec):

* goal **width** = 9.6 Opta units = 7.32 m → posts at Y = 45.2 and 54.8
* goal **height** = 38 Opta units = 2.44 m → crossbar at Z = 38, ground at 0

That frame is consistent with every goal in the file: all 74 sit inside
(Y 45.5–54.4, Z 0.6–34.8) and the tallest is a header at ~2.2 m. It also lines
up with Opta's own `Low*`/`High*` goal-mouth qualifiers, which flip over between
Z = 19.6 and Z = 22.2 (~1.3–1.4 m, i.e. roughly half way up).

> **Caveat:** the crossbar value is a community inference, not a documented
> Stats Perform spec. And `goalMouthZ` for shots that missed *over* the bar is
> not physically credible (close-range headers are logged at Z ≈ 85, ~5 m) —
> only goals are plotted here, so it does not affect the chart.


In [ ]:
def load_goals() -> list[dict[str, str]]:
    with (HERE / "goals.csv").open(encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return [r for r in rows if r["goal_mouth_x"] and r["goal_mouth_y"]]


def to_metres(y_unit: float, z_unit: float) -> tuple[float, float]:
    """Opta goal-mouth units -> metres inside the frame (0,0 = left post, ground)."""
    x = (y_unit - Y_LEFT_POST) / (Y_RIGHT_POST - Y_LEFT_POST) * GOAL_W
    z = (z_unit - Z_GROUND) / (Z_BAR - Z_GROUND) * GOAL_H
    return x, z


def bin_goals(goals: list[dict[str, str]]) -> list[list[int]]:
    """counts[row][col]; row 0 is the top band, col 0 is the shooter's left."""
    counts = [[0] * COLS for _ in range(ROWS)]
    for g in goals:
        x, z = to_metres(float(g["goal_mouth_x"]), float(g["goal_mouth_y"]))
        col = min(COLS - 1, max(0, int(x / (GOAL_W / COLS))))
        row = min(ROWS - 1, max(0, ROWS - 1 - int(z / (GOAL_H / ROWS))))
        counts[row][col] += 1
    return counts


goals = load_goals()
counts = bin_goals(goals)
total = sum(sum(r) for r in counts)

print(f"goals with goal-mouth coordinates: {total}\n")
print("grid (row 0 = top band, col 0 = shooter's left):")
for r, row in enumerate(counts):
    label = ("top   " if r == 0 else "middle" if r == 1 else "bottom")[:6]
    print(f"  {label}  " + "  ".join(f"{c:3d} ({c / total * 100:4.1f}%)" for c in row))

pens = sum(1 for g in goals if g["is_penalty"] == "true")
body: dict[str, int] = {}
for g in goals:
    body[g["body_part"]] = body.get(g["body_part"], 0) + 1
print(f"\nnon-penalties: {total - pens}   penalties: {pens}")
print("body part: " + "   ".join(f"{k}={v}" for k, v in sorted(body.items())))


## Drawing it

Two pieces worth knowing about:

**The body-part silhouettes.** Each icon is designed in its own coordinate
space (head, torso, arms, legs and ball as plain strokes and circles) and then
*fitted uniformly* into a shared box — so all three end up the same height, on
the same centre line, with the same stroke weight, however different their
poses are. The left-foot icon is a horizontal mirror of the right-foot one.

**The summary badges.** The three counts under the goal are flat-top hexagons
(100 × 86.6 units), drawn identically and centred exactly under the goal.


In [ ]:
# --- body-part silhouettes -----------------------------------------------
PT_TO_UNIT = 100.0 / (ICON_SIZE * 72.0)   # points -> icon data units

# lines: (points, stroke width in points)   head/ball: ((x, y), radius)
# strokes are deliberately light so the icons do not read as solid ink blobs
KICKER = (
    [
        ([(44, 52), (41, 28), (42, 12)], 7.5),     # planted leg
        ([(44, 52), (62, 40), (75, 47)], 7.5),     # swinging leg
        ([(36, 77), (44, 52)], 10.5),              # torso
        ([(38, 71), (24, 62)], 6.0),               # trailing arm
        ([(39, 73), (55, 80)], 6.0),               # leading arm
    ],
    ((34, 84), 7.0),                               # head
    ((88, 44), 6.0),                               # ball
)

HEADER = (
    [
        ([(47, 50), (35, 32), (27, 18)], 7.5),     # back leg
        ([(47, 50), (59, 30), (57, 14)], 7.5),     # front leg
        ([(50, 73), (47, 50)], 10.5),              # arched torso
        ([(49, 67), (34, 61)], 6.0),               # lower arm
        ([(50, 69), (64, 77)], 6.0),               # upper arm
    ],
    ((52, 80), 7.4),                               # head
    ((68, 90), 6.3),                               # ball
)


def icon_axes(cx: float, cy: float):
    w, h = ICON_SIZE / FIG_W, ICON_SIZE / FIG_H
    ax = fig.add_axes([cx - w / 2, cy - h / 2, w, h])
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor("none")
    ax.axis("off")
    return ax


def draw_body_icon(cx: float, cy: float, spec, mirror: bool = False) -> None:
    """Fit a figure spec into the icon box, then paint it."""
    lines, head, ball = spec

    xs: list[float] = []
    ys: list[float] = []
    for pts, lw in lines:
        half = lw * PT_TO_UNIT / 2
        for x, y in pts:
            xs += [x - half, x + half]
            ys += [y - half, y + half]
    for (bx, by), r in (head, ball):
        xs += [bx - r, bx + r]
        ys += [by - r, by + r]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)

    scale = min(84.0 / (x1 - x0), 76.0 / (y1 - y0))
    midx, midy = (x0 + x1) / 2, (y0 + y1) / 2

    def T(x: float, y: float) -> tuple[float, float]:
        px = 50 + (x - midx) * scale
        if mirror:
            px = 100 - px
        return px, 50 + (y - midy) * scale

    ax = icon_axes(cx, cy)
    for pts, lw in lines:
        ax.plot([T(x, y)[0] for x, y in pts], [T(x, y)[1] for x, y in pts],
                color=RED, lw=lw, solid_capstyle="round",
                solid_joinstyle="round", zorder=3)
    hx, hy = T(*head[0])
    ax.add_patch(Circle((hx, hy), head[1] * scale, facecolor=RED,
                        edgecolor="none", zorder=4))
    bx, by = T(*ball[0])
    ax.add_patch(Circle((bx, by), ball[1] * scale, facecolor="#FFFFFF",
                        edgecolor=RED, lw=2.4, zorder=5))


def draw_sidebar(body_counts: dict[str, int]) -> None:
    fig.text((SIDE_X + 0.962) / 2, 0.695, "GOALS BY BODY PART", fontsize=12.5,
             fontweight="bold", color=INK_SOFT, ha="center", va="center",
             family=DISPLAY)
    fig.add_artist(
        plt.Line2D([SIDE_X, 0.962], [0.658, 0.658], color=RULE, lw=1.3,
                   transform=fig.transFigure)
    )
    rows = [
        ("header", "Headers", HEADER, False),
        ("foot-left", "Left-Foot", KICKER, True),
        ("foot-right", "Right-Foot", KICKER, False),
    ]
    for cy, (key, label, spec, mirror) in zip(ICON_ROWS, rows):
        draw_body_icon(ICON_CX, cy, spec, mirror)
        fig.text(TEXT_CX, cy + 0.0145, f"{body_counts.get(key, 0)}", fontsize=32,
                 fontweight="medium", color=INK, ha="center", va="center",
                 family=DISPLAY)
        fig.text(TEXT_CX, cy - 0.026, label, fontsize=17, color=INK,
                 ha="center", va="center", family=BODY)


In [ ]:
def draw() -> None:
    total = sum(sum(r) for r in counts)
    shares = [[c / total * 100 for c in row] for row in counts]
    peak = max(max(row) for row in shares)

    body_counts: dict[str, int] = {}
    for g in goals:
        body_counts[g["body_part"]] = body_counts.get(g["body_part"], 0) + 1

    global fig
    fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=DPI, facecolor=BG)
    ax = fig.add_axes([AX_X, AX_Y, AX_W, AX_H])
    ax.set_facecolor(BG)

    # ---- shaded cells --------------------------------------------------
    cw, ch = GOAL_W / COLS, GOAL_H / ROWS
    for r in range(ROWS):                       # row 0 = top band; y grows up
        for c in range(COLS):
            ax.add_patch(Rectangle((c * cw, (ROWS - 1 - r) * ch), cw, ch,
                                   facecolor=CELL_CMAP(shares[r][c] / peak),
                                   edgecolor="none", zorder=2))

    for c in range(1, COLS):                    # dotted separators
        ax.plot([c * cw, c * cw], [0, GOAL_H], color="#FFFFFF",
                lw=1.6, ls=(0, (1.5, 2.4)), zorder=3)
    for r in range(1, ROWS):
        ax.plot([0, GOAL_W], [r * ch, r * ch], color="#FFFFFF",
                lw=1.6, ls=(0, (1.5, 2.4)), zorder=3)

    # ---- cell labels: share of all goals + raw count --------------------
    for r in range(ROWS):
        for c in range(COLS):
            cx = c * cw + cw / 2
            cy = (ROWS - 1 - r) * ch + ch / 2
            colour = "#FFFFFF" if shares[r][c] / peak > 0.72 else INK
            ax.text(cx, cy + 0.165, f"{shares[r][c]:.1f}%", ha="center",
                    va="center", fontsize=17, fontweight="bold",
                    color=colour, zorder=4, family=DISPLAY)
            ax.text(cx, cy - 0.17, f"({counts[r][c]})", ha="center",
                    va="center", fontsize=14, color=colour, zorder=4,
                    family=BODY)

    # ---- goal frame: thin posts, big goal -------------------------------
    lw_post = 7
    ax.plot([0, GOAL_W], [GOAL_H, GOAL_H], color=FRAME, lw=lw_post,
            solid_capstyle="butt", zorder=5)                    # crossbar
    ax.plot([0, 0], [0, GOAL_H], color=FRAME, lw=lw_post,
            solid_capstyle="butt", zorder=5)                    # left post
    ax.plot([GOAL_W, GOAL_W], [0, GOAL_H], color=FRAME, lw=lw_post,
            solid_capstyle="butt", zorder=5)                    # right post
    ax.plot([-X_MARGIN, GOAL_W + X_MARGIN], [0, 0], color=FRAME, lw=lw_post + 1,
            solid_capstyle="butt", zorder=5)                    # ground line

    ax.set_xlim(-X_MARGIN, GOAL_W + X_MARGIN)
    ax.set_ylim(-0.62, GOAL_H + 0.5)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")

    ax.text(-1.32, GOAL_H / 2 + 0.16, "% of Goals in Zone", rotation=90,
            ha="center", va="center", fontsize=15, fontweight="bold",
            color=INK, family=DISPLAY)
    ax.text(-0.98, GOAL_H / 2 - 0.22, "(Goals)", rotation=90, ha="center",
            va="center", fontsize=14, color=INK_SOFT, family=BODY)

    # ---- header --------------------------------------------------------
    fig.text(0.055, 0.905, "Harry Kane Goal-Mouth Placement", fontsize=33,
             fontweight="bold", color=INK, ha="left", va="center", family=DISPLAY)
    fig.text(0.055, 0.856, "All goals 2025/2026 | Bayern Munich & England",
             fontsize=17, color=INK, ha="left", va="center", family=BODY)
    fig.text(0.958, 0.907, "Viz by @oladeeayo", fontsize=16, fontweight="bold",
             color=RED, ha="right", va="center", family=DISPLAY)
    fig.text(0.958, 0.865, "WhoScored & Sofascore event data", fontsize=12,
             color=INK_SOFT, ha="right", va="center", family=BODY)

    draw_sidebar(body_counts)

    # ---- three hexagon badges under the goal ----------------------------
    pens = sum(1 for g in goals if g["is_penalty"] == "true")
    badges = [(f"{total}", "Goals"), (f"{total - pens}", "Non-penalties"),
              (f"{pens}", "Penalties")]
    hex_pts = [(100, 50), (75, 93.3), (25, 93.3),
               (0, 50), (25, 6.7), (75, 6.7)]    # flat-top hexagon

    for cx, (value, label) in zip(BADGE_CX, badges):
        w, h = BADGE_W / FIG_W, BADGE_W / FIG_H  # square box in inches
        box = fig.add_axes([cx - w / 2, BADGE_Y - h / 2, w, h])
        box.set_xlim(0, 100)
        box.set_ylim(0, 100)
        box.set_aspect("equal", adjustable="box")
        box.set_facecolor("none")
        box.axis("off")
        box.add_patch(Polygon(hex_pts, closed=True, facecolor=RED,
                              edgecolor="none", zorder=3))
        box.text(50, 50, value, fontsize=34, fontweight="bold", color="#FFFFFF",
                 ha="center", va="center", family=DISPLAY, zorder=4)
        fig.text(cx, 0.113, label, fontsize=17, color=INK,
                 ha="center", va="center", family=BODY)

    for attempt in range(4):                     # retry: the PNG can be locked
        try:                                     # transiently by an image viewer
            fig.savefig(OUT_PNG, facecolor=BG)
            break
        except OSError:
            if attempt == 3:
                raise
            time.sleep(2)
    fig.savefig(OUT_PDF, facecolor=BG)
    plt.close(fig)
    print(f"wrote {OUT_PNG.name} and {OUT_PDF.name}")


draw()


In [ ]:
display(Image(filename=str(OUT_PNG), width=1100))


## What the data says

* **74 goals** have goal-mouth coordinates — the England–Senegal friendly
  (2025-06-10) is the one goal in `goals.csv` without them.
* The **bottom corners dominate**: 16.2 % bottom-left and 17.6 % bottom-right,
  with a second pocket at middle-right (9.5 %) and a top-right band (8.1 %).
  The very middle of the goal is nearly empty — 0–2.7 % per cell.
* **43 of 74** goals land in the bottom third.
* **20 of the 74 (27 %) are penalties**, which is why they get their own badge —
  they materially shape the pattern.
* Body part: **50** right foot, **14** headers, **10** left foot.

---

**Viz by [@oladeeayo](https://github.com/oladeeayo).**
Data: WhoScored & Sofascore Opta event data, Aug 2025 – Jul 2026.
Fonts: DM Sans & Karla, SIL Open Font License.
